# RTAI-242P | Practical 6 — Model Optimization: Pruning & Quantization

**Objective:** Optimize a pre-trained CNN using pruning and quantization, then compare size, speed, and accuracy.

---
### Techniques Covered
| Technique | What it does | Goal |
|-----------|-------------|------|
| **Pruning** | Zeros out small-magnitude weights | Reduce active parameters |
| **Quantization** | Converts FP32 weights → INT8 | Reduce memory & speed up inference |

## 1. Imports & Setup

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.utils.prune as prune
from torchvision import datasets, transforms
from torch.utils.data import DataLoader
import numpy as np
import copy
import os
import time
import tempfile
import matplotlib.pyplot as plt
import pandas as pd

DEVICE = 'cpu'  # Keep on CPU for quantization compatibility
print(f'PyTorch version : {torch.__version__}')
print(f'Device          : {DEVICE}')

## 2. Load MNIST Dataset

In [ ]:
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.1307,), (0.3081,))
])

train_ds = datasets.MNIST('./data', train=True,  download=True, transform=transform)
test_ds  = datasets.MNIST('./data', train=False, download=True, transform=transform)

train_loader = DataLoader(train_ds, batch_size=64, shuffle=True)
test_loader  = DataLoader(test_ds,  batch_size=64, shuffle=False)

print(f'Train samples : {len(train_ds):,}')
print(f'Test  samples : {len(test_ds):,}')

# Visualise a few samples
imgs, lbls = next(iter(train_loader))
fig, axes = plt.subplots(1, 8, figsize=(12, 2))
for i, ax in enumerate(axes):
    ax.imshow(imgs[i].squeeze(), cmap='gray')
    ax.set_title(lbls[i].item()); ax.axis('off')
plt.suptitle('Sample MNIST Images'); plt.show()

## 3. Define the CNN Model

In [ ]:
class SimpleCNN(nn.Module):
    """2-layer CNN for MNIST classification."""
    def __init__(self):
        super().__init__()
        self.conv1   = nn.Conv2d(1, 16, 3, padding=1)   # 28x28 -> 14x14 after pool
        self.conv2   = nn.Conv2d(16, 32, 3, padding=1)  # 14x14 -> 7x7 after pool
        self.pool    = nn.MaxPool2d(2, 2)
        self.fc1     = nn.Linear(32 * 7 * 7, 128)       # 1568 -> 128
        self.fc2     = nn.Linear(128, 10)                # 128 -> 10 classes
        self.relu    = nn.ReLU()
        self.dropout = nn.Dropout(0.25)

    def forward(self, x):
        x = self.pool(self.relu(self.conv1(x)))
        x = self.pool(self.relu(self.conv2(x)))
        x = x.view(-1, 32 * 7 * 7)
        x = self.relu(self.fc1(x))
        x = self.dropout(x)
        return self.fc2(x)

model = SimpleCNN().to(DEVICE)
print(model)
total_params = sum(p.numel() for p in model.parameters())
print(f'\nTotal parameters: {total_params:,}')

## 4. Helper Functions

In [ ]:
def train_epoch(model, loader, optimizer, criterion):
    model.train()
    loss_sum, correct = 0.0, 0
    for imgs, lbls in loader:
        optimizer.zero_grad()
        out  = model(imgs.to(DEVICE))
        loss = criterion(out, lbls.to(DEVICE))
        loss.backward(); optimizer.step()
        loss_sum += loss.item() * imgs.size(0)
        correct  += (out.argmax(1) == lbls.to(DEVICE)).sum().item()
    return loss_sum / len(loader.dataset), 100.0 * correct / len(loader.dataset)

@torch.no_grad()
def evaluate(model, loader, criterion):
    model.eval()
    loss_sum, correct = 0.0, 0
    for imgs, lbls in loader:
        out  = model(imgs.to(DEVICE))
        loss = criterion(out, lbls.to(DEVICE))
        loss_sum += loss.item() * imgs.size(0)
        correct  += (out.argmax(1) == lbls.to(DEVICE)).sum().item()
    return loss_sum / len(loader.dataset), 100.0 * correct / len(loader.dataset)

def model_size_kb(model):
    with tempfile.NamedTemporaryFile(suffix='.pt', delete=False) as f:
        path = f.name
    torch.save(model.state_dict(), path)
    size = os.path.getsize(path) / 1024
    os.remove(path)
    return size

def avg_inference_ms(model, loader, n=10):
    model.eval()
    times = []
    with torch.no_grad():
        for i, (imgs, _) in enumerate(loader):
            if i >= n: break
            t0 = time.perf_counter()
            model(imgs.to(DEVICE))
            times.append((time.perf_counter() - t0) * 1000)
    return float(np.mean(times))

def get_sparsity(model):
    total = nz = 0
    for p in model.parameters():
        total += p.numel()
        nz    += p.nonzero().size(0)
    return 100.0 * (1 - nz / total)

print('Helper functions defined.')

## 5. Train the Baseline Model

In [ ]:
EPOCHS = 3
optimizer = optim.Adam(model.parameters(), lr=0.001)
criterion = nn.CrossEntropyLoss()

train_accs, val_accs = [], []

for ep in range(1, EPOCHS + 1):
    tr_loss, tr_acc = train_epoch(model, train_loader, optimizer, criterion)
    va_loss, va_acc = evaluate(model, test_loader, criterion)
    train_accs.append(tr_acc); val_accs.append(va_acc)
    print(f'Epoch {ep}/{EPOCHS} | Train Acc: {tr_acc:.2f}% | Val Acc: {va_acc:.2f}%')

torch.save(model.state_dict(), 'baseline_model.pt')

# Plot
plt.figure(figsize=(7,3))
plt.plot(range(1, EPOCHS+1), train_accs, 'b-o', label='Train Acc')
plt.plot(range(1, EPOCHS+1), val_accs,   'r-o', label='Val Acc')
plt.title('Training Progress'); plt.xlabel('Epoch'); plt.ylabel('Accuracy (%)')
plt.legend(); plt.tight_layout(); plt.show()

print(f'\nBaseline  | Accuracy: {val_accs[-1]:.2f}%  Size: {model_size_kb(model):.1f} KB')

## 6. Pruning

**L1-Unstructured Pruning** removes the lowest-magnitude weights across all Conv and Linear layers.

In [ ]:
PRUNE_AMOUNT = 0.50   # Remove 50% of weights

pruned_model = copy.deepcopy(model)   # preserve original

# Apply pruning to each layer
for module, name in [
    (pruned_model.conv1, 'weight'),
    (pruned_model.conv2, 'weight'),
    (pruned_model.fc1,   'weight'),
    (pruned_model.fc2,   'weight'),
]:
    prune.l1_unstructured(module, name=name, amount=PRUNE_AMOUNT)
    prune.remove(module, name)  # Bake mask permanently into weights

_, pruned_acc  = evaluate(pruned_model, test_loader, criterion)
pruned_sparse  = get_sparsity(pruned_model)
pruned_size    = model_size_kb(pruned_model)

print(f'Pruned Model  | Accuracy: {pruned_acc:.2f}%  '
      f'Sparsity: {pruned_sparse:.1f}%  Size: {pruned_size:.1f} KB')

torch.save(pruned_model.state_dict(), 'pruned_model.pt')

# Visualise weight distribution of fc1
w = pruned_model.fc1.weight.detach().numpy().flatten()
plt.figure(figsize=(8, 2.5))
plt.hist(w, bins=120, color='steelblue', edgecolor='white')
plt.axvline(0, color='red', lw=1.5, label='Zero (pruned)')
plt.title(f'fc1 weight distribution after {int(PRUNE_AMOUNT*100)}% pruning')
plt.xlabel('Weight value'); plt.legend(); plt.tight_layout(); plt.show()

## 7. Dynamic Post-Training Quantization

Converts `Linear` layer weights from **FP32 → INT8** — no retraining needed.

In [ ]:
quantized_model = torch.quantization.quantize_dynamic(
    copy.deepcopy(model),   # start from original baseline
    {nn.Linear},            # layers to quantize
    dtype=torch.qint8       # INT8 precision
)

_, quant_acc = evaluate(quantized_model, test_loader, criterion)
quant_size   = model_size_kb(quantized_model)

print(f'Quantized Model | Accuracy: {quant_acc:.2f}%  Size: {quant_size:.1f} KB')
torch.save(quantized_model.state_dict(), 'quantized_model.pt')

## 8. Side-by-Side Comparison

In [ ]:
_, base_acc  = evaluate(model, test_loader, criterion)
base_size    = model_size_kb(model)
base_infer   = avg_inference_ms(model, test_loader)
pruned_infer = avg_inference_ms(pruned_model, test_loader)
quant_infer  = avg_inference_ms(quantized_model, test_loader)

results = pd.DataFrame({
    'Model'          : ['Baseline', 'Pruned', 'Quantized'],
    'Accuracy (%)'   : [round(base_acc, 2), round(pruned_acc, 2), round(quant_acc, 2)],
    'Size (KB)'      : [round(base_size, 1), round(pruned_size, 1), round(quant_size, 1)],
    'Inference (ms)' : [round(base_infer, 2), round(pruned_infer, 2), round(quant_infer, 2)],
    'Sparsity (%)'   : [round(get_sparsity(model), 1), round(pruned_sparse, 1), '-'],
}).set_index('Model')

print('\n', results.to_string())

# Bar chart comparison
fig, axes = plt.subplots(1, 3, figsize=(13, 4))
colors = ['#4C72B0', '#DD8452', '#55A868']
names  = ['Baseline', 'Pruned', 'Quantized']

for ax, col in zip(axes, ['Accuracy (%)', 'Size (KB)', 'Inference (ms)']):
    vals = results[col].values if col != 'Sparsity (%)' else None
    ax.bar(names, results[col].values.astype(float), color=colors)
    ax.set_title(col); ax.set_ylabel(col)
    for i, v in enumerate(results[col].values.astype(float)):
        ax.text(i, v * 0.98, f'{v:.1f}', ha='center', va='top', fontsize=9, color='white', fontweight='bold')

plt.suptitle('Model Optimization — Comparison', fontsize=13, fontweight='bold')
plt.tight_layout(); plt.show()

## 9. Conclusion

| Technique | Size Reduction | Accuracy Impact | Best For |
|-----------|---------------|-----------------|----------|
| **Pruning** | Moderate | Small | Sparse/edge models |
| **Quantization** | ~50-75% | Very small | CPU/mobile inference |
| **Both combined** | Large | Moderate | Ultra-lite deployment |

> Both techniques allow deploying AI models on resource-constrained devices (mobile, IoT, embedded) without significant accuracy loss.